<a href="https://colab.research.google.com/github/hosseinta2/LLM-from-scratch/blob/main/fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():

      print(f"{data_file_path} already exists. Skipping download "
          "and extraction.")
    return

with urllib.request.urlopen(url) as response:
    with open(zip_path, "wb") as out_file:
        out_file.write(response.read())
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extracted_path)

original_file_path = Path(extracted_path) / "SMSSpamCollection"
os.rename(original_file_path, data_file_path)
print(f"File downloaded and saved as {data_file_path}")

File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [135]:
import pandas as pd

df = pd.read_csv(data_file_path,sep="\t", header = None, names = ["Label", "Text"])

num_samples = df[df["Label"]=="spam"].shape[0]
ham_sampled = df[df["Label"]=="ham"].sample(num_samples)
dataset = pd.concat([ham_sampled,df[df["Label"]=="spam"]])

dataset["Label"] = dataset["Label"].map({"ham":0 , "spam":1})

dataset = dataset.sample(num_samples*2).reset_index(drop=True)

idx1 = 1000
idx2= 1200

train_dataset = dataset[:idx1]
val_dataset = dataset[idx1:idx2]
test_dataset = dataset[idx2:]

for x in train_dataset["Text"]:
  print(x)
  break

Why you keeping me away like this


In [46]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

tokenizer.encode("<|endoftext|>",allowed_special={"<|endoftext|>"})

[50256]

In [196]:
import torch
from torch.utils.data import Dataset, DataLoader,TensorDataset

def tokenizer_padding(data,tokenizer,padding_id, max_length=None):
  tokenized_text = []
  tokenized_text2 = []
  label = []
  max_length2 = 0
  for x,y in zip(data["Text"],data["Label"]):
      tokenized_x = tokenizer.encode(x)
      label.append([y])
      max_length2 = max(max_length2,len(tokenized_x))
      tokenized_text.append(tokenized_x)
  if max_length!=None:
      max_length2 = max_length
  for x in tokenized_text:
      if max_length2<len(x):
        x=x[:max_length2]
      else:
        x = x+(max_length2-len(x))*[padding_id]
      tokenized_text2.append(x)
  return torch.tensor(tokenized_text2),torch.tensor(label)

padded_train = tokenizer_padding(train_dataset,tokenizer,50256)
max_length = padded_train[0].shape[1]

padded_val = tokenizer_padding(val_dataset,tokenizer,50256,max_length=max_length)

padded_test = tokenizer_padding(test_dataset,tokenizer,50256,max_length=max_length)



padded_train = TensorDataset(padded_train[0],padded_train[1].squeeze())  # converting the data and labels into the TensorDataset (x,y) suitable for Dataloader
padded_val =   TensorDataset(padded_val[0],padded_val[1].squeeze())
padded_test = TensorDataset(padded_test[0],padded_test[1].squeeze())

training_batches = DataLoader(padded_train,batch_size=8,drop_last=True)
validation_batches = DataLoader(padded_val,batch_size = 8 ,drop_last=False)
test_batches = DataLoader(padded_test,batch_size = 8,drop_last=False)


